# RZ-USFM project · Part 3 — Expert C: Radiomics-Regularized USFM (RZ-USFM)

**Model.** USFM (ViT-B/16, frozen) + rank-8 LoRA on every attention projection. Patch tokens are pooled with the **lesion mask**, the **peritumoral ring** and globally (mask-guided pooling). Two heads:
- **radiomics head** $g_\psi$: predicts the radiomic concepts $\hat{\mathbf r}$ (from Part 1) and their log-variances $s$;
- **task head** (concept bottleneck + side channel): $\ell = \mathbf u^\top\hat{\mathbf r} + \mathbf v^\top P(\mathbf e) + b$.

**Objective**

$$\mathcal L_{\text{RZ-USFM}}=\mathrm{BCE}\big(y,\sigma(\ell)\big)+\lambda\,\frac{1}{2K}\sum_k\Big[\tfrac12(r_k-\hat r_k)^2e^{-s_k}+\tfrac12 s_k\Big]+\mu\,\big\|\mathrm{Cov}(\hat{\mathbf r},P(\mathbf e))\big\|^2$$

$\lambda$ is the **radiomic regularisation weight**; $\lambda=0$ gives plain USFM fine-tuning with the same architecture (**USFM-FT**, the reference model).

**Protocol (no leakage)**
1. **λ grid** for every outer fold (seed 0). λ\* is the best **λ > 0** per fold on a patient-disjoint **validation split of the training data** (the same split as Part 2); λ = 0 is kept as the USFM-FT reference.
2. **Ensemble** of `N_SEEDS` models at λ\* = Expert C (uncertainty = SD of member logits), and the same for λ = 0 (fair ablation and Part 4 ablation).
3. **Radiomic consistency score** $c=(\mathbf r-\hat{\mathbf r})^\top\hat\Sigma^{-1}_{\text{res}}(\mathbf r-\hat{\mathbf r})$, with $\hat\Sigma_{\text{res}}$ estimated on the validation residuals.

**Results:** metrics with 95% CI, DeLong tests, λ curve, concept R², representation probes (frozen USFM vs USFM-FT vs RZ-USFM), consistency score vs errors, per-scanner performance, comparison with Experts A and B, SHAP on the radiomic concepts, Grad-CAM / SHAP / LIME images.

**Requires Parts 1 and 2** (concepts, crop cache) and the USFM weights from your first USFM notebook. Runtime: **T4 GPU**; resumable.

## 1. Setup and configuration

In [ ]:
!pip -q install timm gdown shap lime scikit-image
!git clone -q https://github.com/openmedlab/USFM.git /content/USFM 2>/dev/null || echo 'USFM repo already present'
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, time, json, warnings
warnings.filterwarnings('ignore')

PART1_DIR = '/content/drive/MyDrive/rsusfm_project'                # Part 1: master.csv, common.py, concepts, Expert A
PART3_DIR = '/content/drive/MyDrive/rsusfm_project_part3'          # Part 3: ALL outputs of this notebook
USFM_REPO = '/content/USFM'
USFM_CKPT = '/content/drive/MyDrive/weights/USFM_latest.pth'       # cached by the first USFM notebook
# Part 2 outputs: found automatically in either location
P2_CANDIDATES = ['/content/drive/MyDrive/rsusfm_project_part2/results', f'{PART1_DIR}/results']
P2_RES = next((d for d in P2_CANDIDATES if glob.glob(f'{d}/expertB_crops_*.npz')), None)
assert P2_RES, f'Part 2 outputs (expertB_crops_*.npz) not found in {P2_CANDIDATES} - run Part 2 first'
P1_RES = f'{PART1_DIR}/results'
RUN_DIR, FIG_DIR, RES_DIR = f'{PART3_DIR}/runs', f'{PART3_DIR}/figures', f'{PART3_DIR}/results'
for d in (RUN_DIR, FIG_DIR, RES_DIR): os.makedirs(d, exist_ok=True)
print('Part 1:', PART1_DIR, '| Part 2 results:', P2_RES, '| Part 3 outputs:', PART3_DIR)

IMG, CACHE_SIZE = 224, 288
LAMBDAS   = [0.0, 0.25, 0.5, 1.0, 2.0]   # radiomic regularisation grid (0 = USFM-FT reference)
N_SEEDS   = 3                            # ensemble members for the selected λ* and for λ = 0
ENSEMBLE_BASELINE = True                 # also build the λ = 0 ensemble (fair comparison + Part 4 ablation)
EPOCHS, BS = 15, 32
LR_LORA, LR_HEAD, WD = 5e-4, 1e-3, 0.01
LORA_RANK, LORA_ALPHA = 8, 16
MU_ORTH   = 0.1                          # weight of the concept / side-channel decorrelation term
SEED = 42
# QUICK TEST (about 1 h): LAMBDAS = [0.0, 0.5, 1.0]; N_SEEDS = 1

In [ ]:
sys.path.insert(0, PART1_DIR); sys.path.insert(0, PART3_DIR)
import importlib, common; importlib.reload(common)
from common import *
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, torch.cuda.get_device_name(0) if device == 'cuda' else '')

master = pd.read_csv(f'{PART1_DIR}/master.csv')
y_all, g_all = master.label.values, master.patient.astype(str).values
FOLDS = sorted(master.fold.unique())

cache = f'{P2_RES}/expertB_crops_{CACHE_SIZE}.npz'
assert os.path.exists(cache), 'Crop cache not found - run Part 2 first'
z = np.load(cache, allow_pickle=True)
assert list(z['ids']) == list(master.id), 'crop cache order differs from master.csv'
CROP_IMG, CROP_LES, CROP_RING = z['img'], z['les'], z['ring']

CONCEPTS = {}
for f in FOLDS:
    c = pd.read_csv(f'{PART1_DIR}/radiomics/concepts_fold{f}.csv').set_index('id').loc[master.id]
    CONCEPTS[f] = c.values.astype(np.float32)
CONCEPT_NAMES = list(c.columns); K = len(CONCEPT_NAMES)
print(f'{len(master)} lesions | {K} radiomic concepts:', CONCEPT_NAMES)

## 2. USFM helper and weights

In [ ]:
%%writefile {PART3_DIR}/usfm_colab.py
"""
usfm_colab.py — lightweight helpers to use the USFM foundation model
(Jiao et al., Medical Image Analysis 2024; https://github.com/openmedlab/USFM)
for classification / feature extraction WITHOUT installing mmcv / mmsegmentation.

Only needs: torch, timm, numpy, scipy (all available on Google Colab).
The ViT architecture is imported unchanged from the official repo
(usdsgen/modules/backbone/vision_transformer.py).
"""
import math
import os
import sys
from functools import partial

import numpy as np
import torch
import torch.nn as nn
from scipy.interpolate import RegularGridInterpolator as RGI

# USFM ViT-B configuration (configs/model/Cls/vit.yaml in the official repo)
USFM_VIT_B_CFG = dict(
    img_size=224,
    patch_size=16,
    in_chans=3,
    embed_dim=768,
    depth=12,
    num_heads=12,
    mlp_ratio=4,
    qkv_bias=True,
    attn_drop_rate=0.0,
    drop_path_rate=0.1,
    init_values=0.1,
    use_abs_pos_emb=False,
    use_rel_pos_bias=True,
    use_shared_rel_pos_bias=False,
    use_mean_pooling=True,
)

# USFM downstream preprocessing (usdsgen/data/datasets.py): 224x224, bicubic, ImageNet mean/std
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def _import_vit(repo_dir):
    """Import the official USFM VisionTransformer class from the cloned repo."""
    repo_dir = os.path.abspath(repo_dir)
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    from usdsgen.modules.backbone.vision_transformer import VisionTransformer
    return VisionTransformer


def build_usfm(repo_dir="USFM", num_classes=0, **overrides):
    """Build the USFM ViT-B/16. num_classes=0 -> pure feature extractor (768-d output)."""
    VisionTransformer = _import_vit(repo_dir)
    cfg = {**USFM_VIT_B_CFG, **overrides}
    return VisionTransformer(
        num_classes=num_classes, norm_layer=partial(nn.LayerNorm, eps=1e-6), **cfg
    )


def _interp_rel_pos_table(table, src_size, dst_size, num_extra_tokens):
    """Geometric interpolation of a relative-position-bias table (as in BEiT / USFM)."""
    extra = table[-num_extra_tokens:, :]
    table = table[:-num_extra_tokens, :]

    def gp(a, r, n):
        return a * (1.0 - r**n) / (1.0 - r)

    left, right = 1.01, 1.5
    while right - left > 1e-6:
        q = (left + right) / 2.0
        if gp(1, q, src_size // 2) > dst_size // 2:
            right = q
        else:
            left = q
    dis, cur = [], 1
    for i in range(src_size // 2):
        dis.append(cur)
        cur += q ** (i + 1)
    r_ids = [-d for d in reversed(dis)]
    x = y = r_ids + [0] + dis
    t = dst_size // 2.0
    dx = dy = np.arange(-t, t + 0.1, 1.0)
    xi, yi = np.meshgrid(dx, dy, indexing="ij")
    pts = np.array([xi.ravel(), yi.ravel()]).T
    out = []
    for h in range(table.shape[1]):
        z = table[:, h].view(src_size, src_size).float().numpy()
        f = RGI((x, y), z.T, method="cubic", bounds_error=False)
        out.append(torch.tensor(f(pts).reshape(xi.shape), dtype=torch.float32).reshape(-1, 1))
    return torch.cat([torch.cat(out, dim=-1), extra.float()], dim=0)


def load_usfm_weights(model, ckpt_path, verbose=True):
    """
    Robust loader for USFM_latest.pth.
    Handles: wrapped checkpoints ({'model': ...}), 'module.'/'encoder.' prefixes,
    a shared relative-position bias (expanded to every block), and table-size
    interpolation. (The official remap function raises a KeyError when the shared
    table is absent; this version handles both layouts.)
    Returns a dict with loading statistics.
    """
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    for key in ("model", "state_dict", "module"):
        if isinstance(ckpt, dict) and key in ckpt and isinstance(ckpt[key], dict):
            ckpt = ckpt[key]

    sd = {}
    for k, v in ckpt.items():
        if not torch.is_tensor(v):
            continue
        for p in ("module.", "encoder.", "backbone.", "model."):
            if k.startswith(p):
                k = k[len(p):]
        sd[k] = v

    shared_key = "rel_pos_bias.relative_position_bias_table"
    if shared_key in sd:
        shared = sd.pop(shared_key)
        for i in range(len(model.blocks)):
            sd.setdefault(f"blocks.{i}.attn.relative_position_bias_table", shared.clone())
    sd = {k: v for k, v in sd.items() if "relative_position_index" not in k}

    msd = model.state_dict()
    gh, gw = model.patch_embed.patch_shape
    for k in list(sd.keys()):
        if "relative_position_bias_table" in k and k in msd and sd[k].shape != msd[k].shape:
            num_extra = msd[k].shape[0] - (2 * gh - 1) * (2 * gw - 1)
            src = int(round((sd[k].shape[0] - num_extra) ** 0.5))
            dst = int(round((msd[k].shape[0] - num_extra) ** 0.5))
            sd[k] = _interp_rel_pos_table(sd[k], src, dst, num_extra)

    skipped_shape = [k for k in sd if k in msd and sd[k].shape != msd[k].shape]
    for k in skipped_shape:
        sd.pop(k)
    skipped_head = [k for k in list(sd) if k.startswith("head.")]
    for k in skipped_head:  # never load a pretraining head into a new classifier
        sd.pop(k)

    missing, unexpected = model.load_state_dict(sd, strict=False)
    # relative_position_index buffers are rebuilt at init, so they are not "weights"
    backbone_keys = [k for k in msd if not k.startswith(("head.", "fc_norm."))
                     and "relative_position_index" not in k]
    loaded = [k for k in backbone_keys if k not in missing]
    stats = dict(
        backbone_params=len(backbone_keys),
        loaded=len(loaded),
        missing=[k for k in missing if not k.startswith(("head.", "fc_norm."))
                 and "relative_position_index" not in k],
        unexpected=list(unexpected),
        skipped_shape=skipped_shape,
    )
    if verbose:
        print(f"USFM backbone tensors loaded: {stats['loaded']}/{stats['backbone_params']}")
        if stats["missing"]:
            print("  missing (randomly initialised):", stats["missing"][:10], "...")
        if stats["unexpected"]:
            print("  unexpected (ignored):", stats["unexpected"][:10], "...")
        if skipped_shape:
            print("  skipped (shape mismatch):", skipped_shape[:10])
    if stats["loaded"] < 0.9 * stats["backbone_params"]:
        raise RuntimeError(
            "Less than 90% of USFM backbone weights were loaded - check the checkpoint "
            "file (is it the full USFM_latest.pth and not an HTML page from Google Drive?)."
        )
    return stats


In [ ]:
import gdown
if not os.path.exists(USFM_CKPT) or os.path.getsize(USFM_CKPT) < 50e6:
    os.makedirs(os.path.dirname(USFM_CKPT), exist_ok=True)
    gdown.download(id='1KRwXZgYterH895Z8EpXpR1L1eSMMJo4q', output=USFM_CKPT, quiet=False)
importlib.reload(sys.modules['usfm_colab']) if 'usfm_colab' in sys.modules else None
from usfm_colab import build_usfm, load_usfm_weights
_tmp = build_usfm(USFM_REPO, num_classes=0, img_size=IMG)
load_usfm_weights(_tmp, USFM_CKPT)                       # prints 195/195 if all backbone weights load
BASE_SD = {k: v.clone() for k, v in _tmp.state_dict().items()}
del _tmp

## 3. Data and the RZ-USFM model
- Input = the **whole context crop** (USFM sees lesion, ring and surroundings); the masks are used only for pooling.
- Augmentation is **geometric only** (flip, small affine applied jointly to image and masks): photometric changes would invalidate the radiomic targets, which were computed on the original image.
- LoRA is attached as a weight *parametrization* on `attn.qkv`, because USFM reads `qkv.weight` directly (`F.linear`), so layer-replacing LoRA libraries would silently do nothing.

In [ ]:
import torchvision.transforms.v2 as T2
from torchvision import tv_tensors
from torch.utils.data import Dataset, DataLoader
import torch.nn.utils.parametrize as P
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
GEOM = T2.Compose([T2.RandomHorizontalFlip(), T2.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1))])

class RZDS(Dataset):
    def __init__(self, idx, fold, train): self.idx, self.f, self.train = np.asarray(idx), fold, train
    def __len__(self): return len(self.idx)
    def __getitem__(self, k):
        i = self.idx[k]
        im = tv_tensors.Image(torch.from_numpy(CROP_IMG[i]).float().div(255)[None].repeat(3, 1, 1))
        ml = tv_tensors.Mask(torch.from_numpy(CROP_LES[i])[None]); mr = tv_tensors.Mask(torch.from_numpy(CROP_RING[i])[None])
        if self.train: im, ml, mr = GEOM(im, ml, mr)
        im = F.interpolate(im[None], size=(IMG, IMG), mode='bilinear', align_corners=False)[0]
        ml = F.interpolate(ml[None].float(), size=(IMG, IMG), mode='nearest')[0]
        mr = F.interpolate(mr[None].float(), size=(IMG, IMG), mode='nearest')[0]
        return (im - MEAN) / STD, ml, mr, torch.tensor(float(y_all[i])), torch.from_numpy(CONCEPTS[self.f][i])

class LoRA(nn.Module):
    def __init__(self, out_f, in_f, r=LORA_RANK, alpha=LORA_ALPHA):
        super().__init__()
        self.A = nn.Parameter(torch.randn(r, in_f) * 0.01); self.B = nn.Parameter(torch.zeros(out_f, r))   # B=0: starts as USFM
        self.scale = alpha / r
    def forward(self, W): return W + self.scale * self.B @ self.A

def mask_pool(t, m):                                   # t: (B, N, d) tokens; m: (B, 1, H, W) mask
    a = F.avg_pool2d(m, 16).flatten(1)                 # patch occupancy (B, N)
    return (a.unsqueeze(-1) * t).sum(1) / (a.sum(1, keepdim=True) + 1e-6)

class RZUSFM(nn.Module):
    def __init__(self, n_concepts=K, q=16, lora=True):
        super().__init__()
        self.enc = build_usfm(USFM_REPO, num_classes=0, img_size=IMG)
        self.enc.load_state_dict(BASE_SD)
        for p in self.enc.parameters(): p.requires_grad = False
        if lora:
            for blk in self.enc.blocks:
                P.register_parametrization(blk.attn.qkv, 'weight', LoRA(*blk.attn.qkv.weight.shape))
        d = self.enc.embed_dim
        self.norms = nn.ModuleList([nn.LayerNorm(d) for _ in range(3)])
        self.rad_head = nn.Sequential(nn.Linear(3 * d, 256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256, 2 * n_concepts))
        self.side = nn.Sequential(nn.Dropout(0.2), nn.Linear(3 * d, q))
        self.u = nn.Linear(n_concepts, 1, bias=False); self.v = nn.Linear(q, 1)
        self.K = n_concepts
    def tokens(self, x):
        e = self.enc; x = e.patch_embed(x)
        x = torch.cat([e.cls_token.expand(x.size(0), -1, -1), x], 1)
        for blk in e.blocks: x = blk(x)
        return x[:, 1:]                                  # (B, 196, 768) patch tokens
    def embed(self, x, ml, mr):
        t = self.tokens(x)
        parts = [mask_pool(t, ml), mask_pool(t, mr), t.mean(1)]
        return torch.cat([n(p) for n, p in zip(self.norms, parts)], 1)
    def forward(self, x, ml, mr):
        e = self.embed(x, ml, mr)
        out = self.rad_head(e)
        r_hat, s = out[:, :self.K], out[:, self.K:].clamp(-5, 5)
        z = self.side(e)
        logit = self.u(r_hat).squeeze(1) + self.v(z).squeeze(1)
        return logit, r_hat, s, z, e

def rz_loss(logit, r_hat, s, z, y, r, lam, pos_weight):
    task = F.binary_cross_entropy_with_logits(logit, y, pos_weight=pos_weight)
    rad = (0.5 * (r - r_hat) ** 2 * torch.exp(-s) + 0.5 * s).mean()
    rc, zc = r_hat - r_hat.mean(0), z - z.mean(0)
    orth = ((rc.T @ zc) / max(1, len(z) - 1)).pow(2).mean()
    return task + lam * rad + MU_ORTH * orth, task.detach(), rad.detach()

m = RZUSFM().to(device)
tr = sum(p.numel() for p in m.parameters() if p.requires_grad); tot = sum(p.numel() for p in m.parameters())
with torch.no_grad():
    x, ml, mr, _, _ = next(iter(DataLoader(RZDS(np.arange(4), FOLDS[0], False), batch_size=4)))
    lo, rh, s, zz, e = m(x.to(device), ml.to(device), mr.to(device))
print(f'trainable {tr / 1e6:.2f} M of {tot / 1e6:.1f} M parameters | logit {tuple(lo.shape)} | r_hat {tuple(rh.shape)} | e {tuple(e.shape)}')
del m; torch.cuda.empty_cache()

## 4. Training one model (λ, fold, seed)

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

def val_split(f):                                         # identical to Part 2
    tr = np.where(master.fold != f)[0]
    sgk = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED + f)
    a, b = next(sgk.split(tr, y_all[tr], g_all[tr]))
    return tr[a], tr[b]

def r2_per_concept(r, rh):
    return 1 - ((r - rh) ** 2).sum(0) / (((r - r.mean(0)) ** 2).sum(0) + 1e-8)

@torch.no_grad()
def predict(model, idx, f, tta=True):
    model.eval(); L, R, E = [], [], []
    for x, ml, mr, _, _ in DataLoader(RZDS(idx, f, False), batch_size=64, num_workers=2):
        x, ml, mr = x.to(device), ml.to(device), mr.to(device)
        with torch.autocast('cuda', dtype=torch.float16, enabled=device == 'cuda'):
            lo, rh, _, _, e = model(x, ml, mr)
            if tta:
                lo2, rh2, _, _, _ = model(x.flip(-1), ml.flip(-1), mr.flip(-1)); lo, rh = (lo + lo2) / 2, (rh + rh2) / 2
        L.append(lo.float().cpu()); R.append(rh.float().cpu()); E.append(e.float().cpu())
    return torch.cat(L).numpy(), torch.cat(R).numpy(), torch.cat(E).numpy()

def tag(lam, f, seed): return f'lam{lam:g}_f{f}_s{seed}'

def train_one(lam, f, seed):
    path = f'{RUN_DIR}/{tag(lam, f, seed)}.npz'
    if os.path.exists(path): return dict(np.load(path))
    torch.manual_seed(seed); np.random.seed(seed)
    fit_idx, val_idx = val_split(f); test_idx = np.where(master.fold == f)[0]
    model = RZUSFM().to(device)
    dl = DataLoader(RZDS(fit_idx, f, True), batch_size=BS, shuffle=True, num_workers=2, drop_last=True)
    lora = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith('enc.')]
    head = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith('enc.')]
    opt = torch.optim.AdamW([{'params': lora, 'lr': LR_LORA}, {'params': head, 'lr': LR_HEAD}], weight_decay=WD)
    steps, warm = EPOCHS * len(dl), len(dl)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / warm) * 0.5 * (1 + np.cos(np.pi * min(1, s / steps))))
    pw = torch.tensor((1 - y_all[fit_idx].mean()) / y_all[fit_idx].mean(), device=device)
    scaler = torch.amp.GradScaler(enabled=device == 'cuda')
    curve = []
    for ep in range(EPOCHS):
        model.train(); t0, lt, lr_ = time.time(), 0.0, 0.0
        for x, ml, mr, y, r in dl:
            x, ml, mr, y, r = (a.to(device) for a in (x, ml, mr, y, r))
            with torch.autocast('cuda', dtype=torch.float16, enabled=device == 'cuda'):
                lo, rh, s, zz, _ = model(x, ml, mr)
            loss, l_task, l_rad = rz_loss(lo.float(), rh.float(), s.float(), zz.float(), y, r, lam, pw)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            lt += l_task.item(); lr_ += l_rad.item()
        lv, rv, _ = predict(model, val_idx, f, tta=False)
        auc_v, r2_v = roc_auc_score(y_all[val_idx], lv), r2_per_concept(CONCEPTS[f][val_idx], rv).mean()
        curve.append((lt / len(dl), lr_ / len(dl), auc_v, r2_v))
        print(f'  [λ={lam:g} f{f} s{seed}] ep {ep + 1:2d}/{EPOCHS} task {lt / len(dl):.3f} rad {lr_ / len(dl):.3f} '
              f'val AUC {auc_v:.3f} val R² {r2_v:.3f} ({time.time() - t0:.0f}s)')
    lv, rv, ev = predict(model, val_idx, f); lt_, rt, et = predict(model, test_idx, f)
    out = dict(val_idx=val_idx, val_logit=lv, val_rhat=rv, test_idx=test_idx, test_logit=lt_, test_rhat=rt,
               curve=np.array(curve), val_auc=roc_auc_score(y_all[val_idx], lv), u=model.u.weight.detach().cpu().numpy()[0])
    if seed == SEED: out.update(test_emb=et.astype(np.float16), val_emb=ev.astype(np.float16))
    np.savez(path, **out)
    torch.save({n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad} |
               {n: b.cpu() for n, b in model.named_buffers() if not n.startswith('enc.')}, path.replace('.npz', '.pt'))
    del model, opt; torch.cuda.empty_cache()
    return out

## 5. Phase 1 — λ grid (seed 0) and per-fold selection of λ\* on the validation split
Re-run after a disconnection: finished models are skipped.

In [ ]:
T0 = time.time()
plan = [(lam, f) for f in FOLDS for lam in LAMBDAS]
for k, (lam, f) in enumerate(plan):
    t = time.time(); train_one(lam, f, SEED)
    print(f'>>> {k + 1}/{len(plan)} λ={lam:g} fold {f} ({(time.time() - t) / 60:.1f} min, total {(time.time() - T0) / 60:.0f} min)')

sel = pd.DataFrame([{'fold': f, 'lambda': lam, 'val_AUC': float(np.load(f'{RUN_DIR}/{tag(lam, f, SEED)}.npz')['val_auc'])}
                    for f in FOLDS for lam in LAMBDAS])
pos = sel[sel['lambda'] > 0]                                       # RZ-USFM is defined by λ > 0; λ = 0 is the USFM-FT reference
LAM_STAR = pos.loc[pos.groupby('fold').val_AUC.idxmax()].set_index('fold')['lambda'].to_dict()
display(sel.pivot(index='lambda', columns='fold', values='val_AUC').round(3))
print('selected λ* per fold (best λ > 0 on the validation split, training data only):', LAM_STAR)
best_any = sel.loc[sel.groupby('fold').val_AUC.idxmax()].set_index('fold')['lambda'].to_dict()
print('for reference, best λ including 0:', best_any)
sel.to_csv(f'{RES_DIR}/expertC_lambda_selection.csv', index=False)

## 6. Phase 2 — ensembles at λ\* (Expert C) and at λ = 0 (USFM-FT)

In [ ]:
SEEDS = [SEED + s for s in range(N_SEEDS)]
plan = [(LAM_STAR[f], f, s) for f in FOLDS for s in SEEDS[1:]]
if ENSEMBLE_BASELINE: plan += [(0.0, f, s) for f in FOLDS for s in SEEDS[1:]]
plan = list(dict.fromkeys(plan))          # no duplicates when λ* = 0 for a fold
T0 = time.time()
for k, (lam, f, s) in enumerate(plan):
    t = time.time(); train_one(lam, f, s)
    print(f'>>> {k + 1}/{len(plan)} λ={lam:g} fold {f} seed {s} ({(time.time() - t) / 60:.1f} min, total {(time.time() - T0) / 60:.0f} min)')

## 7. Out-of-fold predictions, uncertainty and the radiomic consistency score
For each fold: ensemble logit mean / SD, Youden threshold on the validation split, ensemble $\hat{\mathbf r}$, and
$c=(\mathbf r-\hat{\mathbf r})^\top\hat\Sigma^{-1}_{\text{res}}(\mathbf r-\hat{\mathbf r})$ with $\hat\Sigma_{\text{res}}$ = Ledoit-Wolf covariance of the **validation** residuals.

In [ ]:
from sklearn.covariance import LedoitWolf

def assemble(lam_of_fold, seeds):
    o = master[['id', 'patient', 'fold', 'label']].copy()
    o[['prob', 'logit_mean', 'logit_sd', 'thr', 'consistency']] = np.nan
    RH, EMB = np.full((len(master), K), np.nan, np.float32), None
    for f in FOLDS:
        runs = [dict(np.load(f'{RUN_DIR}/{tag(lam_of_fold[f], f, s)}.npz')) for s in seeds]
        te, va = runs[0]['test_idx'], runs[0]['val_idx']
        Lt = np.stack([r['test_logit'] for r in runs]); Lv = np.stack([r['val_logit'] for r in runs]).mean(0)
        Rt = np.stack([r['test_rhat'] for r in runs]).mean(0); Rv = np.stack([r['val_rhat'] for r in runs]).mean(0)
        thr = youden_threshold(y_all[va], 1 / (1 + np.exp(-Lv)))
        mu = Lt.mean(0)
        o.loc[te, 'logit_mean'] = mu; o.loc[te, 'prob'] = 1 / (1 + np.exp(-mu))
        o.loc[te, 'logit_sd'] = Lt.std(0) if len(seeds) > 1 else np.nan
        o.loc[te, 'thr'] = thr
        lw = LedoitWolf().fit(CONCEPTS[f][va] - Rv)
        d = CONCEPTS[f][te] - Rt
        o.loc[te, 'consistency'] = np.einsum('ij,jk,ik->i', d, lw.precision_, d)
        RH[te] = Rt
        if 'test_emb' in runs[0]:
            if EMB is None: EMB = np.full((len(master), runs[0]['test_emb'].shape[1]), np.nan, np.float32)
            EMB[te] = runs[0]['test_emb']
    return o, RH, EMB

NAME_RZ, NAME_FT = 'RZ-USFM', 'USFM-FT (λ=0)'
OOF, RHAT, EMBS = {}, {}, {}
OOF[NAME_RZ], RHAT[NAME_RZ], EMBS[NAME_RZ] = assemble(LAM_STAR, SEEDS)
OOF[NAME_FT], RHAT[NAME_FT], EMBS[NAME_FT] = assemble({f: 0.0 for f in FOLDS}, SEEDS if ENSEMBLE_BASELINE else [SEED])
LAM_OOF = {lam: assemble({f: lam for f in FOLDS}, [SEED])[0] for lam in LAMBDAS}      # single-seed λ curve

save_oof(OOF[NAME_RZ], 'expertC', RES_DIR)
save_oof(OOF[NAME_FT], 'USFM-FT', RES_DIR)
extra = OOF[NAME_RZ][['id', 'consistency']].copy(); extra['consistency_log'] = np.log1p(extra.consistency)
extra = pd.concat([extra, pd.DataFrame(RHAT[NAME_RZ], columns=[f'rhat_{c}' for c in CONCEPT_NAMES])], axis=1)
extra.to_csv(f'{RES_DIR}/expertC_extra.csv', index=False)
pd.DataFrame(OOF[NAME_FT][['id', 'consistency']]).to_csv(f'{RES_DIR}/USFM-FT_extra.csv', index=False)
np.save(f'{RES_DIR}/expertC_emb_oof.npy', EMBS[NAME_RZ])
print('saved oof_expertC.csv (Expert C), oof_USFM-FT.csv, expertC_extra.csv (consistency score + predicted concepts)')

## 8. Results — RZ-USFM vs USFM-FT, metrics with 95% CI and DeLong test

In [ ]:
from scipy import stats

def delong(y, p1, p2):
    """Paired DeLong test for two correlated AUCs (Sun & Xu 2014). Returns AUC1, AUC2, p-value."""
    y = np.asarray(y).astype(bool); P = np.vstack([p1, p2]); m, n = y.sum(), (~y).sum()
    def midrank(x):
        o = np.argsort(x); xs = x[o]; r = np.zeros(len(x)); i = 0
        while i < len(x):
            j = i
            while j < len(x) and xs[j] == xs[i]: j += 1
            r[i:j] = 0.5 * (i + j - 1) + 1; i = j
        out = np.empty(len(x)); out[o] = r; return out
    pos, neg = P[:, y], P[:, ~y]
    tx = np.array([midrank(a) for a in pos]); ty = np.array([midrank(a) for a in neg]); tz = np.array([midrank(a) for a in P])
    auc = tz[:, y].sum(1) / m / n - (m + 1) / (2 * n)
    v01 = (tz[:, y] - tx) / n; v10 = 1 - (tz[:, ~y] - ty) / m
    S = np.cov(v01) / m + np.cov(v10) / n
    z = (auc[0] - auc[1]) / np.sqrt(S[0, 0] + S[1, 1] - 2 * S[0, 1] + 1e-12)
    return auc[0], auc[1], 2 * stats.norm.sf(abs(z))

rows = [evaluate_oof(OOF[n], n, RES_DIR) for n in (NAME_RZ, NAME_FT)]
display(summary_table(rows))
pd.DataFrame(rows).to_csv(f'{RES_DIR}/part3_metrics_full.csv', index=False)
a1, a2, p = delong(y_all, OOF[NAME_RZ].prob, OOF[NAME_FT].prob)
print(f'DeLong: RZ-USFM AUC {a1:.3f} vs USFM-FT AUC {a2:.3f}  ->  ΔAUC = {a1 - a2:+.3f}, p = {p:.4f}')

plot_roc(OOF, f'{FIG_DIR}/roc_rzusfm_vs_ft.png', 'RZ-USFM vs USFM fine-tuning (λ = 0)')
plot_pr(OOF, f'{FIG_DIR}/pr_rzusfm_vs_ft.png')
plot_calibration(OOF, f'{FIG_DIR}/calibration_rzusfm_vs_ft.png')
for n in OOF: plot_confusion(OOF[n], n, f'{FIG_DIR}/cm_{n.split()[0]}.png')
plot_metric_bars(rows, f'{FIG_DIR}/metrics_bars_rzusfm_vs_ft.png')

## 9. Comparison of the three experts (A: radiomics, B: deep, C: RZ-USFM)

In [ ]:
ABC = {}
if os.path.exists(f'{P1_RES}/oof_expertA.csv'): ABC['Expert A — radiomics'] = pd.read_csv(f'{P1_RES}/oof_expertA.csv')
if os.path.exists(f'{P2_RES}/oof_expertB.csv'): ABC['Expert B — deep dual-stream'] = pd.read_csv(f'{P2_RES}/oof_expertB.csv')
ABC['Expert C — RZ-USFM'] = OOF[NAME_RZ]
ABC['USFM-FT (λ=0)'] = OOF[NAME_FT]
rows_abc = [evaluate_oof(o, n, RES_DIR) for n, o in ABC.items()]
display(summary_table(rows_abc))
pd.DataFrame(rows_abc).to_csv(f'{RES_DIR}/experts_ABC_metrics.csv', index=False)
plot_roc(ABC, f'{FIG_DIR}/roc_experts_ABC.png', 'Experts A, B, C (out-of-fold)')
plot_calibration(ABC, f'{FIG_DIR}/calibration_experts_ABC.png')
plot_metric_bars(rows_abc, f'{FIG_DIR}/metrics_bars_experts_ABC.png')
names = list(ABC)
tests = [(a, b, *delong(y_all, ABC[a].prob, ABC[b].prob)) for i, a in enumerate(names) for b in names[i + 1:]]
display(pd.DataFrame(tests, columns=['model 1', 'model 2', 'AUC 1', 'AUC 2', 'p (DeLong)']).round(4))

## 10. Effect of the radiomic weight λ (single seed, all folds)

In [ ]:
lam_rows = []
for lam, o in LAM_OOF.items():
    r = evaluate_oof(o, f'lambda_{lam:g}', RUN_DIR, n_boot=300)
    r2 = np.mean([r2_per_concept(CONCEPTS[f][(master.fold == f).values],
                                 np.load(f'{RUN_DIR}/{tag(lam, f, SEED)}.npz')['test_rhat']).mean() for f in FOLDS])
    lo_ci, hi_ci = map(float, r['AUC_CI'].split('-'))
    lam_rows.append({'lambda': lam, 'AUC': r['AUC'], 'lo': lo_ci, 'hi': hi_ci, 'Brier': r['Brier'], 'concept_R2': r2})
lam_df = pd.DataFrame(lam_rows); display(lam_df.round(3)); lam_df.to_csv(f'{RES_DIR}/expertC_lambda_curve.csv', index=False)

fig, ax = plt.subplots(figsize=(6, 4)); x = np.arange(len(lam_df))
ax.errorbar(x, lam_df.AUC, yerr=[lam_df.AUC - lam_df.lo, lam_df.hi - lam_df.AUC], fmt='o-', capsize=3, color='#C44E52', label='AUC (95% CI)')
ax.set_xticks(x, [f'{l:g}' for l in lam_df['lambda']]); ax.set(xlabel='Radiomic regularisation weight λ', ylabel='Out-of-fold AUC')
ax2 = ax.twinx(); ax2.plot(x, lam_df.concept_R2, 's--', color='#4C72B0', label='Concept R² (test)'); ax2.set_ylabel('Radiomic concept R²')
ax.spines['right'].set_visible(True)
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels(); ax.legend(h1 + h2, l1 + l2, loc='lower right', fontsize=8)
ax.set_title('Effect of λ (λ = 0: USFM fine-tuning without radiomics)')
fig.savefig(f'{FIG_DIR}/lambda_curve.png', bbox_inches='tight'); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for lam in LAMBDAS:
    c = np.mean([np.load(f'{RUN_DIR}/{tag(lam, f, SEED)}.npz')['curve'] for f in FOLDS], 0)
    axes[0].plot(c[:, 0], label=f'λ={lam:g}'); axes[1].plot(c[:, 2]); axes[2].plot(c[:, 3])
for a, t in zip(axes, ['Training task loss', 'Validation AUC', 'Validation concept R²']): a.set(title=t, xlabel='Epoch')
axes[0].legend(fontsize=8); fig.tight_layout(); fig.savefig(f'{FIG_DIR}/learning_curves_lambda.png', bbox_inches='tight'); plt.show()

## 11. Does radiomic regularisation put radiomic information into USFM? (concept R² and probes)
- **Concept R²**: how well RZ-USFM's radiomics head predicts each measured concept on unseen lesions.
- **Linear probe**: ridge regression from the pooled embedding $\mathbf e$ to the concepts, fitted on the validation split and tested on the test fold, for **frozen USFM**, **USFM-FT (λ=0)** and **RZ-USFM** — a fair before/after comparison of the representations.

In [ ]:
from sklearn.linear_model import RidgeCV
r2_rz = pd.DataFrame([r2_per_concept(CONCEPTS[f][(master.fold == f).values], RHAT[NAME_RZ][(master.fold == f).values]) for f in FOLDS],
                     columns=CONCEPT_NAMES)

@torch.no_grad()
def frozen_embeddings():
    m = RZUSFM(lora=False).to(device).eval(); E = []
    for x, ml, mr, _, _ in DataLoader(RZDS(np.arange(len(master)), FOLDS[0], False), batch_size=64, num_workers=2):
        with torch.autocast('cuda', dtype=torch.float16, enabled=device == 'cuda'):
            t = m.tokens(x.to(device))
            E.append(torch.cat([mask_pool(t, ml.to(device)), mask_pool(t, mr.to(device)), t.mean(1)], 1).float().cpu())
    del m; return torch.cat(E).numpy()
E_frozen = frozen_embeddings()

def probe(get_val_test):
    out = []
    for f in FOLDS:
        (Ev, rv), (Et, rt) = get_val_test(f)
        reg = RidgeCV(alphas=np.logspace(-1, 4, 12)).fit(Ev, rv)
        out.append(r2_per_concept(rt, reg.predict(Et)))
    return pd.DataFrame(out, columns=CONCEPT_NAMES)

def from_runs(lam_of_fold):
    def g(f):
        r = np.load(f'{RUN_DIR}/{tag(lam_of_fold[f], f, SEED)}.npz')
        return (r['val_emb'].astype(np.float32), CONCEPTS[f][r['val_idx']]), (r['test_emb'].astype(np.float32), CONCEPTS[f][r['test_idx']])
    return g
def from_frozen(f):
    va = val_split(f)[1]; te = np.where(master.fold == f)[0]
    return (E_frozen[va], CONCEPTS[f][va]), (E_frozen[te], CONCEPTS[f][te])

PROBES = {'USFM frozen': probe(from_frozen), 'USFM-FT (λ=0)': probe(from_runs({f: 0.0 for f in FOLDS})),
          'RZ-USFM': probe(from_runs(LAM_STAR))}
tab = pd.DataFrame({k: v.mean() for k, v in PROBES.items()}); tab['RZ-USFM head'] = r2_rz.mean()
display(tab.round(3)); tab.to_csv(f'{RES_DIR}/expertC_concept_R2.csv')
print('mean R² across concepts:', tab.mean().round(3).to_dict())

fig, ax = plt.subplots(figsize=(8, 0.42 * K + 1.5)); yy = np.arange(K); w = 0.2
for j, col in enumerate(tab.columns):
    ax.barh(yy + (j - 1.5) * w, tab[col].clip(lower=-0.2), w, label=col)
ax.set_yticks(yy, CONCEPT_NAMES, fontsize=8); ax.axvline(0, c='k', lw=.8)
ax.set(xlabel='R² on unseen lesions', title='How much radiomic information each representation carries'); ax.legend(fontsize=8)
fig.savefig(f'{FIG_DIR}/concept_R2_probes.png', bbox_inches='tight'); plt.show()

## 12. Radiomic consistency score: does it flag unreliable predictions?
If $c$ is high, RZ-USFM's view of the tissue disagrees with the measured radiomics. This should coincide with errors and with high ensemble uncertainty; Part 4 uses it to gate the fusion.

In [ ]:
o = OOF[NAME_RZ].copy()
o['error'] = ((o.prob >= o.thr).astype(int) != o.label).astype(int); o['c_log'] = np.log1p(o.consistency)
auc_c = roc_auc_score(o.error, o.c_log)
auc_u = roc_auc_score(o.error, o.logit_sd) if o.logit_sd.notna().all() else np.nan
rho = stats.spearmanr(o.c_log, o.logit_sd).correlation if o.logit_sd.notna().all() else np.nan
print(f'AUROC for detecting RZ-USFM errors:  consistency score c = {auc_c:.3f} | ensemble SD = {auc_u:.3f}   (0.5 = no information)')
print(f'Spearman correlation between c and ensemble SD: {rho:.3f}  (low = complementary reliability signals)')

q = pd.qcut(o.c_log, 5, labels=[f'Q{i}' for i in range(1, 6)])
err_by_q = o.groupby(q).error.mean()
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].boxplot([o.c_log[o.error == 0], o.c_log[o.error == 1]]); axes[0].set_xticks([1, 2], ['correct', 'error'])
axes[0].set(ylabel='log(1 + c)', title=f'Consistency score vs errors (AUROC = {auc_c:.2f})')
axes[1].bar(err_by_q.index.astype(str), err_by_q.values, color='#C44E52')
axes[1].set(xlabel='Consistency-score quintile (Q5 = least consistent)', ylabel='Error rate', title='Error rate by consistency quintile')
fig.tight_layout(); fig.savefig(f'{FIG_DIR}/consistency_score.png', bbox_inches='tight'); plt.show()

## 13. Performance by scanner (robustness)

In [ ]:
if 'device' in master:
    rows_dev = []
    for dev in sorted(master.device.dropna().unique()):
        msk = (master.device == dev).values
        if len(np.unique(y_all[msk])) < 2: continue
        row = {'device': dev, 'n': int(msk.sum()), 'malignant %': 100 * y_all[msk].mean()}
        for n, oo in ABC.items(): row[n] = roc_auc_score(y_all[msk], oo.prob.values[msk])
        rows_dev.append(row)
    dev_df = pd.DataFrame(rows_dev).set_index('device'); display(dev_df.round(3))
    dev_df.to_csv(f'{RES_DIR}/auc_by_scanner.csv')
    ax = dev_df[list(ABC)].plot.bar(figsize=(8, 4), rot=0)
    ax.set(ylabel='AUC', ylim=(0.4, 1.0), title='AUC by ultrasound scanner (out-of-fold)'); ax.legend(fontsize=8)
    plt.savefig(f'{FIG_DIR}/auc_by_scanner.png', bbox_inches='tight'); plt.show()
    print('Note: this is within-dataset subgroup analysis; the leave-one-scanner-out experiment is run in Part 4.')

## 14. Interpretability through the radiomic concepts (exact SHAP)
The decision is $\ell=\mathbf u^\top\hat{\mathbf r}+\mathbf v^\top P(\mathbf e)+b$, which is **linear in the predicted concepts**, so the exact SHAP value of concept $k$ for lesion $i$ is $u_k(\hat r_{ik}-\bar{\hat r}_k)$. We also report how much of the logit variance goes through the radiomic-concept path vs the side channel.

In [ ]:
import shap
SV, X, PATH = [], [], []
for f in FOLDS:
    r = np.load(f'{RUN_DIR}/{tag(LAM_STAR[f], f, SEED)}.npz')
    u, rh = r['u'], r['test_rhat']
    sv = u * (rh - rh.mean(0)); SV.append(sv); X.append(rh)
    concept_part = rh @ u; PATH.append(np.var(concept_part) / (np.var(r['test_logit']) + 1e-8))
SV, X = np.vstack(SV), np.vstack(X)
expl = shap.Explanation(values=SV, data=X, feature_names=CONCEPT_NAMES, base_values=np.zeros(len(SV)))
plt.figure(); shap.plots.beeswarm(expl, max_display=K, show=False)
plt.title('RZ-USFM: SHAP values of the radiomic concepts (exact, linear bottleneck)')
plt.savefig(f'{FIG_DIR}/shap_concepts_beeswarm.png', bbox_inches='tight'); plt.show()
plt.figure(); shap.plots.bar(expl, max_display=K, show=False)
plt.savefig(f'{FIG_DIR}/shap_concepts_bar.png', bbox_inches='tight'); plt.show()
pd.DataFrame({'concept': CONCEPT_NAMES, 'mean_abs_shap': np.abs(SV).mean(0)}).sort_values('mean_abs_shap', ascending=False) \
  .to_csv(f'{RES_DIR}/expertC_concept_shap.csv', index=False)
print(f'Share of logit variance carried by the radiomic-concept path (mean over folds): {np.mean(PATH):.2f}')

## 15. Image explanations on fold-0 test cases (TP, TN, FP, FN)
Token Grad-CAM on the last USFM block, SHAP (partition explainer, blur masker) and LIME (SLIC superpixels), for the fold-0 RZ-USFM model (seed 0).

In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries, slic

f0, lam0 = FOLDS[0], LAM_STAR[FOLDS[0]]
model = RZUSFM().to(device).eval()
model.load_state_dict(torch.load(f'{RUN_DIR}/{tag(lam0, f0, SEED)}.pt', map_location=device), strict=False)
o0 = assemble({f: LAM_STAR[f] for f in FOLDS}, [SEED])[0]
te0 = np.where(master.fold == f0)[0]; pred = (o0.prob >= o0.thr).astype(int).values
picks = {}
for name, cond in {'TP': (y_all == 1) & (pred == 1), 'TN': (y_all == 0) & (pred == 0),
                   'FP': (y_all == 0) & (pred == 1), 'FN': (y_all == 1) & (pred == 0)}.items():
    cand = [i for i in te0 if cond[i]]
    if cand: picks[name] = cand[int(np.argmax(np.abs(o0.logit_mean.values[cand])))]
print('cases:', {k: master.id[v] for k, v in picks.items()})

def tensors(i):
    x, ml, mr, _, _ = RZDS([i], f0, False)[0]
    return x[None].to(device), ml[None].to(device), mr[None].to(device)
def to_rgb(x): return (x[0].cpu() * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy()

acts = {}
hook = model.enc.blocks[-1].register_forward_hook(lambda m, i, o: acts.__setitem__('t', o))
def token_gradcam(i):
    x, ml, mr = tensors(i)
    with torch.enable_grad():
        lo = model(x, ml, mr)[0]; t = acts['t']
        g = torch.autograd.grad(lo.sum(), t)[0][:, 1:]; a = t[:, 1:]
        cam = F.relu((g.mean(1, keepdim=True) * a).sum(-1))[0]
    s = int(np.sqrt(cam.numel()))
    cam = F.interpolate(cam.view(1, 1, s, s), size=(IMG, IMG), mode='bilinear')[0, 0].detach().cpu().numpy()
    return cam / (cam.max() + 1e-8), torch.sigmoid(lo).item()

def make_f(i):
    _, ml, mr = tensors(i)
    @torch.no_grad()
    def f(batch):
        out = []
        for s in range(0, len(batch), 32):
            b = torch.tensor(np.asarray(batch[s:s + 32]), dtype=torch.float32, device=device).permute(0, 3, 1, 2)
            b = (b - MEAN.to(device)) / STD.to(device); n = len(b)
            with torch.autocast('cuda', dtype=torch.float16, enabled=device == 'cuda'):
                lo = model(b, ml.expand(n, -1, -1, -1), mr.expand(n, -1, -1, -1))[0]
            out.append(torch.sigmoid(lo.float()).cpu().numpy())
        p = np.concatenate(out); return np.stack([1 - p, p], 1)
    return f

masker = shap.maskers.Image('blur(16,16)', (IMG, IMG, 3)); lime_exp = lime_image.LimeImageExplainer(random_state=SEED)
fig, axes = plt.subplots(len(picks), 4, figsize=(12, 3 * len(picks))); axes = np.atleast_2d(axes)
for r_, (name, i) in enumerate(picks.items()):
    x, ml, mr = tensors(i); rgb = to_rgb(x); cam, p = token_gradcam(i); f = make_f(i)
    sv = shap.Explainer(lambda b: f(b)[:, 1:], masker)(rgb[None], max_evals=1000, batch_size=50)
    s = sv.values[0, ..., 0].sum(-1); lim = np.abs(s).max() + 1e-8
    e = lime_exp.explain_instance(rgb, f, labels=(1,), hide_color=0, num_samples=500,
                                  segmentation_fn=lambda im: slic(im, n_segments=50, compactness=10, start_label=1, channel_axis=-1))
    tmp, msk = e.get_image_and_mask(1, positive_only=False, num_features=6, hide_rest=False)
    axes[r_, 0].imshow(rgb); axes[r_, 0].contour(ml[0, 0].cpu(), [0.5], colors='r', linewidths=1); axes[r_, 0].contour(mr[0, 0].cpu(), [0.5], colors='c', linewidths=.7)
    axes[r_, 0].set_title(f'{name}: {["benign", "malignant"][y_all[i]]}, p = {p:.2f}', fontsize=9)
    axes[r_, 1].imshow(rgb); axes[r_, 1].imshow(cam, cmap='jet', alpha=.45); axes[r_, 1].set_title('Token Grad-CAM', fontsize=9)
    axes[r_, 2].imshow(rgb, alpha=.6); axes[r_, 2].imshow(s, cmap='bwr', vmin=-lim, vmax=lim, alpha=.6)
    axes[r_, 2].contour(ml[0, 0].cpu(), [0.5], colors='k', linewidths=.8); axes[r_, 2].set_title('SHAP (red → malignant)', fontsize=9)
    axes[r_, 3].imshow(mark_boundaries(tmp, msk)); ov = np.zeros((*msk.shape, 4)); ov[msk == 1] = [0, 1, 0, .35]; ov[msk == -1] = [1, 0, 0, .35]
    axes[r_, 3].imshow(ov); axes[r_, 3].set_title('LIME (green → malignant)', fontsize=9)
for a in axes.ravel(): a.axis('off')
fig.suptitle(f'RZ-USFM explanations (fold {f0}, λ* = {lam0:g})', fontsize=10); fig.tight_layout()
fig.savefig(f'{FIG_DIR}/xai_rzusfm_cases.png', bbox_inches='tight'); plt.show()
hook.remove()

# predicted vs measured concept profile for the same cases
fig, axes = plt.subplots(1, len(picks), figsize=(4 * len(picks), 3.6), sharey=True); axes = np.atleast_1d(axes)
for ax, (name, i) in zip(axes, picks.items()):
    ax.barh(np.arange(K) + .2, CONCEPTS[f0][i], .4, label='measured (PyRadiomics)')
    ax.barh(np.arange(K) - .2, RHAT[NAME_RZ][i], .4, label='predicted (RZ-USFM)')
    ax.set_title(f'{name} — c = {OOF[NAME_RZ].consistency.values[i]:.1f}', fontsize=9)
axes[0].set_yticks(np.arange(K), CONCEPT_NAMES, fontsize=7); axes[0].legend(fontsize=7)
fig.tight_layout(); fig.savefig(f'{FIG_DIR}/concepts_measured_vs_predicted_cases.png', bbox_inches='tight'); plt.show()

## 16. Outputs of Part 3 (all in `rsusfm_project_part3/`)
| File | Used by |
|---|---|
| `results/oof_expertC.csv` | **Part 4 fusion** (RZ-USFM ensemble: prob, logit mean, logit SD, threshold) |
| `results/expertC_extra.csv` | **Part 4 fusion**: consistency score $c$, $\log(1+c)$, predicted concepts |
| `results/oof_USFM-FT.csv`, `USFM-FT_extra.csv` | Part 4 ablation: fusion with plain USFM instead of RZ-USFM |
| `results/expertC_lambda_selection.csv`, `expertC_lambda_curve.csv` | λ selection (validation) and λ sensitivity |
| `results/expertC_concept_R2.csv`, `expertC_concept_shap.csv`, `auc_by_scanner.csv`, `experts_ABC_metrics.csv` | paper tables |
| `runs/*.pt` | LoRA + heads of every model (small files) |
| `figures/*.png` | paper figures (300 dpi) |